### Going back to our model

In [79]:
import pandas as pd
import numpy as np

df = pd.read_csv('../module3/data-week-3.csv')

df.columns = df.columns.str.lower().str.replace(' ', '_')

categoricals = list(df.dtypes[df.dtypes == 'str'].index)

for c in categoricals:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0) 

df.churn = (df.churn == 'yes').astype(int)
df.churn.mean()

np.float64(0.2653698707936959)

Split

In [80]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

del df_train['churn']
del df_val['churn']
del df_test['churn']

One Hot Encoding

In [81]:
from sklearn.feature_extraction import DictVectorizer

categoricals = list(df.dtypes[df.dtypes == 'str'].index)
categoricals.remove('customerid')
print(categoricals)
numericals = list(df.dtypes[df.dtypes != 'str'].index)
numericals.remove('churn')
print(numericals)


dv = DictVectorizer(sparse=False)
train_dicts = df_train[categoricals + numericals].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categoricals + numericals].to_dict(orient='records')
X_val = dv.transform(val_dicts)

test_dicts = df_test[categoricals + numericals].to_dict(orient='records')
X_test = dv.transform(test_dicts)


['gender', 'partner', 'dependents', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod']
['seniorcitizen', 'tenure', 'monthlycharges', 'totalcharges']


In [82]:
y_train.mean(), y_val.mean(), y_test.mean()

(np.float64(0.2686390532544379),
 np.float64(0.2739531582682754),
 np.float64(0.24698367636621718))

Predictions

In [83]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)


y_pred = model.predict_proba(X_val)[:, 1]
churn_decision = (y_pred >= 0.5)
(y_val == churn_decision).mean()

/workspaces/ML-zoomcamp/zoomcamp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


np.float64(0.8034066713981547)